# EDA — Bureau & Bureau Balance

Reference: PLAN_v2.md §2.2 — `bureau_balance` is 27M rows; Polars streaming required.

**Checks performed:**
1. Row counts and join cardinality (one-to-many at both levels).
2. `bureau_balance` STATUS distribution.
3. `CREDIT_ACTIVE` and `CREDIT_TYPE` distributions (drives stratified aggregations).
4. Per-borrower bureau-record counts.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import polars as pl
import matplotlib.pyplot as plt
import seaborn as sns

from src.data import read_raw

sns.set_style('whitegrid')

In [ ]:
bureau = read_raw('bureau')
bb = read_raw('bureau_balance')
print(f'bureau         : {bureau.shape}')
print(f'bureau_balance : {bb.shape}')

## 1. Join cardinality

**Expectation**: each `SK_ID_CURR` has multiple `SK_ID_BUREAU`s (mean ~5–6); each `SK_ID_BUREAU` has many monthly balance rows.

In [ ]:
per_curr = bureau.group_by('SK_ID_CURR').len().rename({'len': 'n_bureau_records'})
print(f'unique SK_ID_CURR in bureau: {per_curr.height:,}')
print(f'mean records per borrower : {per_curr["n_bureau_records"].mean():.2f}')
print(f'max records per borrower  : {per_curr["n_bureau_records"].max()}')

per_bureau = bb.group_by('SK_ID_BUREAU').len().rename({'len': 'n_balance_rows'})
print(f'\nunique SK_ID_BUREAU in balance: {per_bureau.height:,}')
print(f'mean balance rows per bureau  : {per_bureau["n_balance_rows"].mean():.2f}')

## 2. STATUS distribution

In [ ]:
bb['STATUS'].value_counts().sort('count', descending=True)

**Codes**: `0` = no DPD, `1`–`5` = months past due (severity), `C` = closed, `X` = unknown.

## 3. CREDIT_ACTIVE & CREDIT_TYPE

In [ ]:
print(bureau['CREDIT_ACTIVE'].value_counts().sort('count', descending=True))
print()
print(bureau['CREDIT_TYPE'].value_counts().sort('count', descending=True).head(10))